In [44]:
import random
import json

In [4]:
dataset = json.load(open("locomo10.json", "r"))

In [39]:
def format_session(session, timestamp):
    session_conv = f"TIMESTAMP: {timestamp}\nCONVERSATION:\n"
    for dialogue in session:
        utterance = ""
        if dialogue.get("blip_caption") is not None:
            utterance += f"<image_description>{dialogue['blip_caption']}</image_description>"
        utterance += dialogue['text']
        session_conv += f"\t{dialogue['speaker']}: {utterance}\n"
    return session_conv + '\n'

In [40]:
def format_single_hop(evidence, conversation):
    session_id = int(evidence.split(':')[0][-1])
    session = conversation.get(f"session_{session_id}")
    session_date = conversation.get(f"session_{session_id}_date_time")
    context_conversation = format_session(session, session_date)
    return context_conversation

In [41]:
def format_long_horizon(conversation):
    session_ids = [item for i in range(len(conversation)) for item in conversation.keys() if item == 'session_%d' % (i+1)]
    full_conv = ""
    for session_id in session_ids:
        session_date = conversation[f"{session_id}_date_time"]
        session = conversation[session_id]
        session_conv = format_session(session, session_date)
        full_conv += session_conv 
    return full_conv

In [43]:
PROMPT = """Answer the question based on the following conversation.
{CONVERSATION}

Question: {QUESTION}

Answer with exact words from the conversations whenever possible and do not write lengthy and complete sentences.
If the answer cannot be found, do not share false information.
"""

In [ ]:
def format_dataset(examples):
    qas = examples['qa']
    conversation = examples['conversation']
    
    for qa in qas:
        category = qa['category']
        if category == 4:
            context_conv = format_single_hop(qa['evidence'], conversation)
        else:
            context_conv = format_long_horizon(conversation)
        query = qa['question']
        
        if category == 2:
            query += " Use the timestamp of the conversation to answer with an approximate date."
        elif category == 5:
            choices = "(a) {}\n(b){}"
            